In [1]:
# First, let's examine the structure of one parquet file to understand the data format
import pandas as pd
import numpy as np
import os
from glob import glob

# # Load one sample file to understand the structure
# sample_file = "/home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/results/trials/epoch_0_item_0/noise_0/train/neuron_input_batch_0/metrics.parquet"
# sample_df = pd.read_parquet(sample_file)

# print("Sample data structure:")
# print(f"Shape: {sample_df.shape}")
# print(f"Columns: {sample_df.columns.tolist()}")
# print("\nFirst few rows:")
# print(sample_df.head())
# print("\nData types:")
# print(sample_df.dtypes)
# print(f"\nUnique stimulus_ids: {sample_df['stimulus_id'].nunique()}")
# print(f"Unique neuron_ids: {sample_df['neuron_id'].nunique()}")
# print(f"Unique labels: {sample_df['label'].unique()}")

Methods:
I will train an unsupervised hierarchical SNN on 60,000 MNIST images. 
I have recorded the weights of the network at 0, 20,000, 40,000 and 60,000 images.
I recorded the spike counts of each final layer neuron in the network during a testing round where for each weight set I displayed the same 60,000 images with either 0,5,15,30, or 50 noise level.

I want to explore two hypotheses: is network able to extract meaningful features at higher noise levels when trained on low noise levels; are the features observed at 0 noise robust to the introduction of noise.
I will test the first hypothesis by training a  linear classifier on the spike counts of each final layer neuron during a test round of the same 60,000 images, using k-fold training so I dont need a validation set, under each noise and checkpoint condition (seeing if liquid generalises)

I will test the second hypothesis by taking the weight vector for the classifier trained at 0 noise for each checkpoint and testing against higher noise levels (seeing if classifier generalises)

So I want a bit of code that loads in the data, and returns X and y.

Then I want a bit of code that takes X and y and returns results (including a weight matrix)
Then I want a bit of code that takes X and y and a weight vector and gives accuracy.

Then I want a bit of code that takes a list of tuples and returns a list of accuracy.

Then I want a bit of code that takes a tuple with training X and y as well as a list of tuples that are test X and ys



In [2]:
import numpy as np
import pandas as pd

import os
from glob import glob

def load_dataset(checkpoint, noise_level, split='train', feature_type='count') -> tuple:
    """
    This function:
    * loads a dataset from a specified checkpoint and noise level
    * pivots it so that rows are stimulus no and columns are neuron number
    * returns that table along with the labels for each
    
    Args:
        checkpoint: 0, 20000, 40000, or 60000
        noise_level: 0, 5, 15, 30, or 50
        split: 'train' or 'test' 
        feature_type: 'count', 'latency', or 'both'
    
    Returns:
        X: DataFrame with shape (n_stimuli, n_neurons) or (n_stimuli, 2*n_neurons) if feature_type='both'
        y: Series with labels for each stimulus
    """
    
    # Construct the path to the data
    base_path = "/home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/results/trials"
    data_path = f"{base_path}/epoch_0_item_{checkpoint}/noise_{noise_level}/{split}"
    
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"Path does not exist: {data_path}")
    
    # Get all batch directories
    batch_dirs = glob(os.path.join(data_path, "neuron_input_batch_*"))
    batch_dirs = sorted(batch_dirs, key=lambda x: int(x.split('_')[-1]))
    
    print(f"Loading {len(batch_dirs)} batches for checkpoint {checkpoint}, noise {noise_level}, split {split}")
    
    # Load all parquet files and concatenate
    dfs = []
    for batch_dir in batch_dirs:
        parquet_file = os.path.join(batch_dir, "metrics.parquet")
        if os.path.exists(parquet_file):
            df = pd.read_parquet(parquet_file)
            # Add batch offset to stimulus_id to make them globally unique
            batch_num = int(batch_dir.split('_')[-1])
            df['stimulus_id'] = df['stimulus_id'] + (batch_num * 50)  # Each batch has 50 stimuli
            dfs.append(df)
    
    if not dfs:
        raise FileNotFoundError(f"No parquet files found in {data_path}")
    
    # Concatenate all batches
    full_df = pd.concat(dfs, ignore_index=True)
    
    print(f"Total stimuli: {full_df['stimulus_id'].nunique()}")
    print(f"Total neurons: {full_df['neuron_id'].nunique()}")
    
    # Create pivot tables based on feature type
    if feature_type == 'count':
        X = full_df.pivot(index='stimulus_id', columns='neuron_id', values='count')
        X.columns = [f'neuron_{i}_count' for i in X.columns]
        
    elif feature_type == 'latency':
        X = full_df.pivot(index='stimulus_id', columns='neuron_id', values='latency')
        X.columns = [f'neuron_{i}_latency' for i in X.columns]
        
    elif feature_type == 'both':
        count_df = full_df.pivot(index='stimulus_id', columns='neuron_id', values='count')
        latency_df = full_df.pivot(index='stimulus_id', columns='neuron_id', values='latency')
        
        count_df.columns = [f'neuron_{i}_count' for i in count_df.columns]
        latency_df.columns = [f'neuron_{i}_latency' for i in latency_df.columns]
        
        X = pd.concat([count_df, latency_df], axis=1)
    else:
        raise ValueError("feature_type must be 'count', 'latency', or 'both'")
    
    # Get labels (same for all neurons, so just take the first occurrence of each stimulus)
    y = full_df.groupby('stimulus_id')['label'].first().sort_index()
    
    # Ensure X and y have the same index
    X = X.sort_index()
    y = y.sort_index()
    
    print(f"Final X shape: {X.shape}")
    print(f"Final y shape: {y.shape}")
    print(f"Labels distribution: {y.value_counts().sort_index().to_dict()}")
    
    return X, y

In [3]:
# # Test the load_dataset function
# print("Testing load_dataset function...")
# try:
#     X_test, y_test = load_dataset(checkpoint=0, noise_level=0, split='train')
#     print(f"Successfully loaded data: X shape {X_test.shape}, y shape {y_test.shape}")
#     print(f"Sample X data (first 5 rows, first 5 columns):")
#     print(X_test.iloc[:5, :5])
#     print(f"Sample y data (first 10 labels): {y_test.head(10).tolist()}")
# except Exception as e:
#     print(f"Error loading dataset: {e}")

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Classifier generalization analysis
# This tests if classifiers trained on clean data (noise=0) can generalize to noisy data
checkpoints = [0, 20000, 40000, 60000]
noise_levels = [0, 5, 15, 30, 50]

generalisation_results_df = pd.DataFrame(index=checkpoints, columns=noise_levels, dtype=float)

for checkpoint in checkpoints:    
    print(f"\n=== Processing checkpoint {checkpoint} ===")
    
    # Train classifier on clean data (noise=0)
    X, y = load_dataset(checkpoint, 0, split='train')
    
    from sklearn.model_selection import cross_val_score
    from sklearn.linear_model import LogisticRegression
    
    print("Training classifier on clean data...")
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
    scores = cross_val_score(clf, X, y, cv=5)
    print("Cross-validation scores:", scores)
    
    # Fit on full clean dataset
    clf.fit(X, y)
    print("Model trained successfully.")
    
    # Test on clean data first
    clean_score = clf.score(X, y)
    generalisation_results_df.loc[checkpoint, 0] = clean_score
    print(f"Clean data accuracy: {clean_score:.4f}")
    
    # Now test on each noise level
    for noise_level in noise_levels[1:]:  # Skip noise=0 since we already did it
        print(f"  Testing on noise level {noise_level}...")
        try:
            X_noise, y_noise = load_dataset(checkpoint, noise_level, split='train')
            score = clf.score(X_noise, y_noise)
            generalisation_results_df.loc[checkpoint, noise_level] = score
            print(f"  Noise {noise_level} accuracy: {score:.4f}")
        except Exception as e:
            print(f"  Error loading noise {noise_level}: {e}")
            generalisation_results_df.loc[checkpoint, noise_level] = np.nan

print(f"\n=== Final Results ===")
print(generalisation_results_df)

# Save results
generalisation_results_df.to_csv("classifier_generalisation_matrix.csv")
print("Saved as classifier_generalisation_matrix.csv")


=== Processing checkpoint 0 ===
Loading 1200 batches for checkpoint 0, noise 0, split train
Total stimuli: 60000
Total neurons: 4096
Final X shape: (60000, 4096)
Final y shape: (60000,)
Labels distribution: {0: 5923, 1: 6742, 2: 5958, 3: 6131, 4: 5842, 5: 5421, 6: 5918, 7: 6265, 8: 5851, 9: 5949}
Training classifier on clean data...
Cross-validation scores: [0.71466667 0.72116667 0.71558333 0.71583333 0.72208333]
Model trained successfully.
Clean data accuracy: 0.9964
  Testing on noise level 5...
Loading 1200 batches for checkpoint 0, noise 5, split train
Total stimuli: 60000
Total neurons: 4096
Final X shape: (60000, 4096)
Final y shape: (60000,)
Labels distribution: {0: 5923, 1: 6742, 2: 5958, 3: 6131, 4: 5842, 5: 5421, 6: 5918, 7: 6265, 8: 5851, 9: 5949}
  Noise 5 accuracy: 0.4340
  Testing on noise level 15...
Loading 1200 batches for checkpoint 0, noise 15, split train
Total stimuli: 60000
Total neurons: 4096
Final X shape: (60000, 4096)
Final y shape: (60000,)
Labels distributi

In [ ]:
import matplotlib.pyplot as plt

# Make sure the index and columns are in correct numerical order
results_df = generalisation_results_df.sort_index()
results_df = results_df[sorted(results_df.columns, key=int)]

# Plot
plt.figure(figsize=(10, 6))

for checkpoint in results_df.index:
    plt.plot(
        results_df.columns.astype(int),        # Noise levels (x-axis)
        results_df.loc[checkpoint],            # Accuracies for this checkpoint (y-axis)
        marker='o',
        label=f'Checkpoint {checkpoint}',
        linewidth=2,
        markersize=6
    )

plt.xlabel("Noise Level (σ)")
plt.ylabel("Accuracy")
plt.title("Classifier Generalisation Across Noise Levels\n(Trained on Clean Data, Tested on Noisy Data)")
plt.legend(title="SNN Checkpoint")
plt.grid(True, alpha=0.3)
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig("classifier_generalisation_plot.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Liquid generalization analysis 
# This tests how well the SNN features perform at each noise level when trained on that same noise level
checkpoints = [0, 20000, 40000, 60000]
noise_levels = [0, 5, 15, 30, 50]

rows = []
print("=== Liquid Generalization Analysis ===")

for checkpoint in checkpoints:
    print(f"\nProcessing checkpoint {checkpoint}...")
    
    for noise_level in noise_levels:
        print(f"  Noise level {noise_level}...")
        
        try:
            X, y = load_dataset(checkpoint, noise_level, split='train')
            
            from sklearn.model_selection import cross_val_score
            from sklearn.linear_model import LogisticRegression
            
            clf = LogisticRegression(max_iter=1000, random_state=42)
            scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
            
            # Also fit and get training score
            clf.fit(X, y)
            train_score = clf.score(X, y)
            
            result = {
                'checkpoint': checkpoint,
                'noise_level': noise_level,
                'cv_mean': scores.mean(),
                'cv_std': scores.std(),
                'train_score': train_score
            }
            
            rows.append(result)
            print(f"    CV Score: {scores.mean():.4f} ± {scores.std():.4f}")
            
        except Exception as e:
            print(f"    Error: {e}")
            rows.append({
                'checkpoint': checkpoint,
                'noise_level': noise_level,
                'cv_mean': np.nan,
                'cv_std': np.nan,
                'train_score': np.nan
            })

results_df = pd.DataFrame(rows)

# Save to CSV
results_df.to_csv("liquid_generalisation_results.csv", index=False)
print(f"\nSaved results to liquid_generalisation_results.csv")
print(f"Results shape: {results_df.shape}")
print("\nSample results:")
print(results_df.head(10))

In [ ]:
import matplotlib.pyplot as plt

# Ensure noise_level is numeric and sorted properly
results_df['noise_level'] = results_df['noise_level'].astype(int)
results_df = results_df.sort_values(by=['checkpoint', 'noise_level'])

# Create plot
plt.figure(figsize=(8, 6))

for checkpoint in results_df['checkpoint'].unique():
    df_subset = results_df[results_df['checkpoint'] == checkpoint]
    plt.plot(
        df_subset['noise_level'],
        df_subset['cv_mean'],
        marker='o',
        label=f'Checkpoint {checkpoint}'
    )
    # Optional: add error bars
    plt.fill_between(
        df_subset['noise_level'],
        df_subset['cv_mean'] - df_subset['cv_std'],
        df_subset['cv_mean'] + df_subset['cv_std'],
        alpha=0.2
    )

plt.xlabel("Noise Level (σ)")
plt.ylabel("Mean CV Accuracy")
plt.title("Classifier Generalisation vs Noise")
plt.legend(title="SNN Checkpoint")
plt.grid(True)
plt.tight_layout()
plt.savefig("generalisation_plot.png")
plt.show()


In [ ]:
import numpy as np
import pandas as pd

def compute_specific_information_wide(X: pd.DataFrame,
                                       y: pd.Series,
                                       n_bins: int = 3) -> pd.DataFrame:
    """
    Compute specific information I_spec(c; r) for each neuron and each class label,
    given X (spike counts: rows=stimuli, cols=neurons) and y (class label per stimulus).

    Returns a DataFrame of shape (n_neurons, n_classes) with bits of info.
    """
    neurons = X.columns
    classes = np.unique(y)
    N = len(y)

    # P(c) overall
    p_c = y.value_counts().sort_index() / N

    # Prepare output
    spec_info = pd.DataFrame(index=neurons, columns=classes, dtype=float)

    for neuron in neurons:
        counts = X[neuron].values
        # define equally‑spaced bin edges
        bins = np.linspace(counts.min(), counts.max(), n_bins + 1)
        # assign each count to bin 0..n_bins-1
        bin_idx = np.digitize(counts, bins[1:-1])
        
        # overall P(r=b)
        p_r = pd.Series(bin_idx).value_counts().sort_index() / N
        
        # P(r=b | c)
        # build a DataFrame for convenience
        df_nr = pd.DataFrame({
            'class': y.values,
            'bin': bin_idx
        })
        p_r_c = df_nr.groupby(['class', 'bin']).size()\
                     .div(df_nr.groupby('class').size())\
                     .unstack(fill_value=0)

        # now compute I_spec for each class
        for c in classes:
            info = 0.0
            for b in range(n_bins):
                pr_c = p_r_c.loc[c, b] if b in p_r_c.columns else 0.0
                pr = p_r.get(b, 0.0)
                if pr_c > 0 and pr > 0:
                    # via Bayes: log2 [P.class|r / P.class] == log2[P(r|c)/P(r)]
                    info += pr_c * np.log2(pr_c / pr)
            spec_info.loc[neuron, c] = info

    return spec_info

In [ ]:
def plot_spec_info(spec_info: pd.DataFrame, checkpoint: int, noise_level: int):
    """
    Plots max spec info for each neuron focusing on a log scale of rank on the x and informativity on the y-axis.
    """
    import matplotlib.pyplot as plt

    # Prepare data for plotting
    sorted_info = spec_info.sort_values(ascending=False)
    ranks = np.log10(np.arange(1, len(sorted_info) + 1))

    # Create the plot
    plt.figure(figsize=(10, 6))
    plt.plot(ranks, sorted_info, marker='o')
    plt.title(f"Max Specific Information (Checkpoint: {checkpoint}, Noise Level: {noise_level})")
    plt.xlabel("Log10(Rank)")
    plt.ylabel("Informativity (bits)")
    plt.grid(True)
    plt.show()

In [ ]:
# Specific Information Analysis
# This analyzes how informative each neuron is for classification across different conditions
print("=== Specific Information Analysis ===")

for checkpoint in checkpoints:
    print(f"\nProcessing checkpoint {checkpoint}...")
    
    for noise_level in noise_levels:
        print(f"  Computing specific information for noise level {noise_level}...")
        
        try:
            X, y = load_dataset(checkpoint, noise_level, split='train')
            
            # Compute specific information
            spec_info = compute_specific_information_wide(X, y, n_bins=3)
            
            # Get max specific information per neuron (most informative class for each neuron)
            max_spec_info_per_neuron = spec_info.max(axis=1)
            
            # Plot the results
            plot_spec_info(max_spec_info_per_neuron, checkpoint, noise_level)
            
            # Save the specific information data
            spec_info.to_csv(f"specific_info_checkpoint_{checkpoint}_noise_{noise_level}.csv")
            max_spec_info_per_neuron.to_csv(f"max_specific_info_checkpoint_{checkpoint}_noise_{noise_level}.csv")
            
            print(f"    Mean max spec info: {max_spec_info_per_neuron.mean():.4f}")
            print(f"    Std max spec info: {max_spec_info_per_neuron.std():.4f}")
            print(f"    Top 10 most informative neurons: {max_spec_info_per_neuron.nlargest(10).index.tolist()}")
            
        except Exception as e:
            print(f"    Error: {e}")

## Summary: How to Use the Data Loading Functions

The `load_dataset()` function is your main interface to the data. Here's how to use it:

### Basic Usage:
```python
# Load spike count data for training
X, y = load_dataset(checkpoint=0, noise_level=0, split='train', feature_type='count')

# Load latency data  
X, y = load_dataset(checkpoint=20000, noise_level=15, split='train', feature_type='latency')

# Load both spike counts and latencies
X, y = load_dataset(checkpoint=60000, noise_level=50, split='train', feature_type='both')
```

### Parameters:
- **checkpoint**: 0, 20000, 40000, or 60000 (training stage)
- **noise_level**: 0, 5, 15, 30, or 50 (noise added to input)
- **split**: 'train' or 'test' 
- **feature_type**: 'count' (spike counts), 'latency' (first spike times), or 'both'

### Data Structure:
- **X**: DataFrame with shape (n_stimuli, n_neurons) 
  - Rows = MNIST images (60,000 for train, 10,000 for test)
  - Columns = Final layer neurons (4,096 neurons)
  - Values = Spike counts, latencies, or both
- **y**: Series with MNIST labels (0-9) for each stimulus

### Available Analyses:
1. **Classifier Generalization**: Train on clean data, test on noisy data
2. **Liquid Generalization**: Train and test on same noise level  
3. **Specific Information**: Measure how informative each neuron is

All analysis functions are ready to run once you have the data loaded!